# 03 — Enrollment: Embedding VGGFace2 → Centroid Gallery**Tujuan:** mengubah setiap wajah ter-augmentasi menjadi **embedding 512-dim** (InceptionResnetV1 pretrained VGGFace2), lalu menghitung **satu centroid per orang** dari data Train. Hasilnya adalah gallery yang dipakai notebook 04 (kalibrasi), 05 (evaluasi), dan 06 (kamera live).**Kenapa tidak dilatih (training)?** PRD §9: arsitektur baru = *embedding + cosine similarity*, bukan classifier. Menambah orang baru cukup menambah centroid — **tanpa training ulang** (FR-4).**Prasyarat** — `02_augmentasi_massal.ipynb` sudah dijalankan:- `Kapita_aug\augmented\<orang>\<grup>_orig.jpg` + `_aug00..24.jpg`- `Kapita_aug\augmented_manifest.csv`- `Kapita_aug\split_assignment.csv` ← **sumber kebenaran split****Urutan eksekusi:** SEL 1 → SEL 9, berurutan, dari kernel kosong.**Output (artefak):**| Artefak | Isi ||---|---|| `Kapita\gallery.npz` | centroid L2-normalized (69 × 512), daftar person, flag `aktif` || `Kapita\gallery_meta.json` | nama, NIM, kelas, jumlah foto latih, status aktif per orang || `Kapita_aug\embeddings_cache.npz` | cache embedding **semua** 17.680 file (dipakai notebook 04 & 05 agar cepat) |**Normalisasi input (PRD §9.2 & §11):** gambar sudah 160×160 ter-align → `(pixel − 127.5) / 128`, sama persis dengan yang dipakai notebook 01 dan yang akan dipakai notebook 06.

In [ ]:
# SEL 1 — Impor paket + konstanta
import os
import sys
import re
import json
import time
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn.functional as F
import pillow_heif
from PIL import Image
pillow_heif.register_heif_opener()

from facenet_pytorch import InceptionResnetV1
import matplotlib
import matplotlib.pyplot as plt

# --- struktur direktori (PRD §20.B) ---
DATA_ROOT = r"./data_riset/raw"
MODEL_DIR = os.path.join(DATA_ROOT, "model")
AUG_ROOT = r"./data_riset/aug"
AUG_MANIFEST = os.path.join(AUG_ROOT, "augmented_manifest.csv")
ALIGN_MANIFEST = os.path.join(AUG_ROOT, "aligned_manifest.csv")
SPLIT_FILE = os.path.join(AUG_ROOT, "split_assignment.csv")
GALLERY_NPZ = os.path.join(DATA_ROOT, "gallery.npz")
GALLERY_META = os.path.join(DATA_ROOT, "gallery_meta.json")
CACHE_NPZ = os.path.join(AUG_ROOT, "embeddings_cache.npz")

# --- parameter enrollment (PRD §9) ---
MODEL_NAME = "vggface2"      # InceptionResnetV1 pretrained VGGFace2
EMB_DIM = 512
IMG_SIZE = 160
BATCH_SIZE = 64
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)

for p in (DATA_ROOT, AUG_ROOT, MODEL_DIR):
    os.makedirs(p, exist_ok=True)

print("Python        :", sys.version.split()[0])
print("torch         :", torch.__version__, "| device:", DEVICE)
print("AUG_MANIFEST  :", os.path.isfile(AUG_MANIFEST))
print("SPLIT_FILE    :", os.path.isfile(SPLIT_FILE))

In [ ]:
# SEL 2 — Baca manifest + terapkan split dari split_assignment.csv (sumber kebenaran)


def imread_unicode(path):
    """Baca gambar RGB uint8 (aman path non-ASCII + fallback HEIC)."""
    img = None
    try:
        img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    except Exception:
        img = None
    if img is None:
        try:
            with Image.open(path) as pil:
                img = cv2.cvtColor(np.array(pil.convert("RGB")), cv2.COLOR_RGB2BGR)
        except Exception:
            return None
    return img


assert os.path.isfile(AUG_MANIFEST), "jalankan 02_augmentasi_massal.ipynb lebih dulu"
assert os.path.isfile(SPLIT_FILE), "split_assignment.csv tidak ditemukan"

df_aug = pd.read_csv(AUG_MANIFEST, encoding="utf-8-sig")
df_split = pd.read_csv(SPLIT_FILE, encoding="utf-8-sig")
df_asli = pd.read_csv(ALIGN_MANIFEST, encoding="utf-8-sig")[["person", "grup", "nama_file"]]

n_awal = len(df_aug)
# Sumber kebenaran = split_assignment.csv; gabungkan ke manifest
df = df_aug.drop(columns=["split"]).merge(df_split, on=["person", "grup"], how="left", validate="many_to_one")
assert df["split"].notna().all(), "ada baris tanpa split -> split_assignment.csv tidak lengkap"
# nama file asli dipakai untuk memulihkan NIM / kelas yang tidak ada di nama folder
df = df.merge(df_asli, on=["person", "grup"], how="left", validate="many_to_one")
assert len(df) == n_awal, "jumlah baris berubah setelah merge"
assert df["nama_file"].notna().all(), "nama_file asli tidak ditemukan untuk semua grup"

print("baris manifest        :", len(df))
print("kelas/orang           :", df["person"].nunique())
print("grup foto asli        :", df.groupby(["person", "grup"]).ngroups)
print("\nfile per split:")
print(df.groupby("split").size().to_string())
print("\nfile per varian (top 5):")
print(df["varian"].value_counts().head(5).to_string())

df_train = df[df["split"] == "train"].copy()
per_orang = df_train.groupby("person")["grup"].nunique()
print("\nkelas dengan data Train:", per_orang.shape[0], "/ 69")
print("grup Train per kelas   : min {} maks {}".format(int(per_orang.min()), int(per_orang.max())))
print("file Train per kelas   : min {} maks {}".format(
    int(df_train.groupby("person").size().min()), int(df_train.groupby("person").size().max())))
assert per_orang.shape[0] == 69, "semua orang harus punya data Train"
print("\nOK: manifest & split konsisten.")

In [ ]:
# SEL 3 — Metadata orang: nama, NIM, kelas (dari nama folder + nama file asli)
KELAS_DIKENAL = ("SK7A", "SKU7A", "SKPROF", "SKALIHPROGRAM", "P1")


def normalisasi_kelas_token(tok):
    """'SK PROF' / 'SKProf' / 'SK Profesional' -> SKPROF, 'SKAlihProgram' -> SKALIHPROGRAM."""
    t = tok.upper().replace(" ", "").replace("-", "").replace("_", "")
    if "PROF" in t:
        return "SKPROF"
    if "ALIHPROGRAM" in t:
        return "SKALIHPROGRAM"
    if t in ("SK7A", "SKU7A", "P1", "SKP1"):
        return "P1" if t in ("P1", "SKP1") else t
    return ""


def kelas_dari_teks(teks):
    u = teks.upper().replace(" ", "")
    for k in ("SKPROFESIONAL", "SKALIHPROGRAM", "SKPROF", "SKU7A", "SK7A", "P1"):
        if k in u:
            return normalisasi_kelas_token(k)
    return ""


def parse_person(folder):
    """'nama_nim_kelas' -> (nama, nim, kelas). NIM = token angka >= 6 digit."""
    nim, kelas, nama_tokens = "", "", []
    for t in [x for x in folder.split("_") if x.strip()]:
        if not nim and t.isdigit() and len(t) >= 6:
            nim = t
            continue
        k = normalisasi_kelas_token(t)
        if k and not kelas:
            kelas = k
            continue
        nama_tokens.append(t.strip())
    return " ".join(nama_tokens).strip(), nim, kelas


baris_meta = []
for person, sub in df.groupby("person"):
    nama, nim, kelas = parse_person(person)
    # fallback: pulihkan NIM & kelas dari nama file foto asli bila folder tidak memuatnya
    if nim == "":
        ditemukan = sorted(set(re.findall(r"\b\d{8,15}\b", " ".join(sub["nama_file"].astype(str)))))
        if ditemukan:
            nim = ditemukan[0]
    if kelas == "":
        dari_file = sorted(set(k for k in (kelas_dari_teks(f) for f in sub["nama_file"].astype(str)) if k))
        if dari_file:
            kelas = dari_file[0]
    baris_meta.append({"person": person, "nama": nama, "nim": nim,
                       "kelas": kelas if kelas else "-",
                       "n_file_latih": int((sub["split"] == "train").sum()),
                       "n_grup_latih": int(sub[sub["split"] == "train"]["grup"].nunique()),
                       "aktif": True})

df_meta = pd.DataFrame(baris_meta).sort_values("person").reset_index(drop=True)

print("jumlah orang       :", len(df_meta))
print("NIM terisi         : {}/69".format(int((df_meta["nim"] != "").sum())))
print("nama terisi        : {}/69".format(int((df_meta["nama"] != "").sum())))
print("kelas terisi       : {}/69".format(int((df_meta["kelas"] != "-").sum())))
print("\ndistribusi kelas:")
print(df_meta["kelas"].value_counts().to_string())
print("\ncontoh 8 baris:")
print(df_meta.head(8).to_string(index=False))

if (df_meta["nim"] == "").any():
    print("\nPERINGATAN: NIM kosong untuk:", df_meta.loc[df_meta["nim"] == "", "person"].tolist())
if (df_meta["kelas"] == "-").any():
    print("\nkelas tidak diketahui (dikosongkan '-'):", df_meta.loc[df_meta["kelas"] == "-", "person"].tolist())
assert df_meta["nim"].is_unique or True  # NIM boleh kosong, tapi dicek uniqueness bila ada
dup_nim = df_meta[df_meta["nim"] != ""]["nim"].duplicated().sum()
print("\nNIM duplikat        :", int(dup_nim), "(harus 0)")

In [ ]:
# SEL 4 — Muat model embedding + fungsi embed batch (KANONIK, dipakai juga di nb 05/06)
print("memuat InceptionResnetV1(pretrained='{}') ...".format(MODEL_NAME))
net = InceptionResnetV1(pretrained=MODEL_NAME).eval().to(DEVICE)
for prm in net.parameters():
    prm.requires_grad_(False)          # hanya inferensi, hemat VRAM 2 GB


def preprocess_embedding(bgr):
    """BGR uint8 160x160 -> tensor CHW ternormalisasi (PRD §9.2: (x - 127.5) / 128)."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x = rgb.astype(np.float32)
    x = (x - 127.5) / 128.0
    return torch.from_numpy(x).permute(2, 0, 1)


@torch.no_grad()
def embed_batch(paths):
    """List path -> (n, 512) embedding L2-normalized di CPU."""
    batch = []
    for p in paths:
        im = imread_unicode(p)
        if im is None:
            continue
        if im.shape[0] != IMG_SIZE or im.shape[1] != IMG_SIZE:
            im = cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        batch.append(preprocess_embedding(im))
    if not batch:
        return np.zeros((0, EMB_DIM), dtype=np.float32)
    x = torch.stack(batch).to(DEVICE)
    emb = net(x)
    emb = F.normalize(emb, dim=1)      # normalisasi ulang (aman)
    return emb.detach().cpu().numpy().astype(np.float32)


# uji smoke: 8 gambar pertama harus menghasilkan embedding valid
uji = embed_batch(df["path_augmented"].head(8).tolist())
norma = np.linalg.norm(uji, axis=1)
print("embedding uji       :", uji.shape, "dtype", uji.dtype)
print("norma L2 (harus ~1) :", np.round(norma, 4).tolist())
assert uji.shape == (8, EMB_DIM), "dimensi embedding salah"
assert np.allclose(norma, 1.0, atol=1e-3), "embedding belum L2-normalized"
print("OK: model & preprocessing embedding siap.")

In [ ]:
# SEL 5 — Hitung embedding untuk SELURUH file (17.680), simpan cache
paths = df["path_augmented"].tolist()
n_total = len(paths)
embedding = np.zeros((n_total, EMB_DIM), dtype=np.float32)
gagal = []

t_start = time.time()
for awal in range(0, n_total, BATCH_SIZE):
    potong = paths[awal:awal + BATCH_SIZE]
    emb = embed_batch(potong)
    if emb.shape[0] != len(potong):
        # ada file yang gagal dibaca -> isi sisanya dengan embedding tak valid (akan dibuang nanti)
        for i in range(awal, awal + len(potong)):
            gagal.append(paths[i])
        emb = np.vstack([emb, np.zeros((len(potong) - emb.shape[0], EMB_DIM), dtype=np.float32)])
    embedding[awal:awal + len(potong)] = emb
    selesai = awal + len(potong)
    if selesai % 1280 == 0 or selesai == n_total:
        elapsed = time.time() - t_start
        eta = elapsed / selesai * (n_total - selesai)
        print("  [{}/{}] {:5.1f}%  elapsed {:.1f}s  ETA {:.1f}s".format(
            selesai, n_total, 100.0 * selesai / n_total, elapsed, eta))

durasi = time.time() - t_start
print("\nselesai dalam {:.1f} detik ({:.2f} ms/file)".format(durasi, 1000.0 * durasi / n_total))
print("file gagal dibaca    :", len(gagal))
if gagal:
    for g in gagal[:10]:
        print("   -", g)

norma = np.linalg.norm(embedding, axis=1)
valid = norma > 0.5
print("embedding valid      : {}/{}".format(int(valid.sum()), n_total))
assert valid.sum() > 0.95 * n_total, "terlalu banyak embedding tidak valid"

np.savez_compressed(CACHE_NPZ,
                    embedding=embedding,
                    person=df["person"].to_numpy().astype("U"),
                    grup=df["grup"].to_numpy().astype("U"),
                    varian=df["varian"].to_numpy().astype("U"),
                    split=df["split"].to_numpy().astype("U"),
                    path=df["path_augmented"].to_numpy().astype("U"),
                    model=np.array([MODEL_NAME]),
                    img_size=np.array([IMG_SIZE]))
print("cache disimpan       :", CACHE_NPZ)

In [ ]:
# SEL 6 — Bangun gallery: centroid per orang dari split Train (FR-4: tanpa training ulang)
df["emb"] = list(embedding)          # kolom embedding sementara
df_train = df[df["split"] == "train"].copy()

person_ids = sorted(df_train["person"].unique())
centroid = np.zeros((len(person_ids), EMB_DIM), dtype=np.float32)
statistik = []

for i, person in enumerate(person_ids):
    sub = df_train[df_train["person"] == person]
    e = np.stack([v for v in sub["emb"] if np.linalg.norm(v) > 0.5])
    c = e.mean(axis=0)
    c = c / np.linalg.norm(c)             # centroid harus L2-normalized
    centroid[i] = c
    # kualitas: similarity dalam orang vs ke centroid orang lain
    dalam = e @ c
    mask = np.ones(len(person_ids), dtype=bool)
    mask[i] = False
    ke_lain = e @ centroid[mask].T
    statistik.append({"person": person, "n_train": int(len(e)),
                      "mean_dalam": float(dalam.mean()), "min_dalam": float(dalam.min()),
                      "max_ke_lain": float(ke_lain.max())})

df_stat = pd.DataFrame(statistik)
norma_c = np.linalg.norm(centroid, axis=1)
print("jumlah centroid      :", centroid.shape)
print("norma centroid       : min {:.4f} maks {:.4f}".format(norma_c.min(), norma_c.max()))
print("\nrata-rata similarity Train ke centroid sendiri:")
print("  mean : {:.4f}".format(df_stat["mean_dalam"].mean()))
print("  min  : {:.4f}".format(df_stat["min_dalam"].min()))
print("  maks similarity ke centroid orang lain: {:.4f}".format(df_stat["max_ke_lain"].max()))

df_meta_g = df_meta.set_index("person").loc[person_ids].reset_index()
aktif = df_meta_g["aktif"].to_numpy().astype(bool)

np.savez_compressed(GALLERY_NPZ,
                    embeddings=centroid.astype(np.float32),
                    person=np.array(person_ids, dtype=object).astype("U"),
                    aktif=aktif,
                    dim=np.array([EMB_DIM]),
                    model=np.array([MODEL_NAME]),
                    img_size=np.array([IMG_SIZE]),
                    dibuat=np.array([datetime.now().strftime("%Y-%m-%d %H:%M:%S")], dtype="U"))

meta = {
    "dibuat": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL_NAME,
    "embedding_dim": EMB_DIM,
    "img_size": IMG_SIZE,
    "normalisasi": "(pixel - 127.5) / 128",
    "split_sumber": "train",
    "jumlah_orang": len(person_ids),
    "jumlah_aktif": int(aktif.sum()),
    "orang": [
        {
            "person": r["person"],
            "nama": r["nama"],
            "nim": r["nim"],
            "kelas": r["kelas"],
            "n_file_latih": int(r["n_file_latih"]),
            "n_grup_latih": int(r["n_grup_latih"]),
            "aktif": bool(r["aktif"]),
        }
        for _, r in df_meta_g.iterrows()
    ],
}
with open(GALLERY_META, "w", encoding="utf-8") as fh:
    json.dump(meta, fh, ensure_ascii=False, indent=2)

print("\ntersimpan:")
print("  gallery.npz        :", GALLERY_NPZ)
print("  gallery_meta.json  :", GALLERY_META)
print("\n5 orang dengan 'mean_dalam' terendah (kandidat perlu foto tambahan):")
print(df_stat.nsmallest(5, "mean_dalam").to_string(index=False))

In [ ]:
# SEL 7 — Verifikasi gallery + diagnosis (shape, norma, sebaran, pasangan mirip)
g = np.load(GALLERY_NPZ, allow_pickle=False)
emb = g["embeddings"]
person_ids = g["person"].tolist()
aktif = g["aktif"]

print("gallery.npz keys     :", g.files)
print("bentuk embeddings    :", emb.shape, "dtype", emb.dtype)
print("jumlah person        :", len(person_ids), "| unik:", len(set(person_ids)))
print("aktif                :", int(aktif.sum()), "/", len(aktif))
print("norma baris          : min {:.4f} maks {:.4f}".format(
    float(np.linalg.norm(emb, axis=1).min()), float(np.linalg.norm(emb, axis=1).max())))
print("nilai rentang        : {:.3f} s.d. {:.3f}".format(float(emb.min()), float(emb.max())))

with open(GALLERY_META, encoding="utf-8") as fh:
    meta = json.load(fh)
print("\nmetadata orang       :", meta["jumlah_orang"], "| aktif:", meta["jumlah_aktif"])
print("contoh metadata      :", json.dumps(meta["orang"][0], ensure_ascii=False))

# similarity antar centroid -> pasangan yang paling mirip (risiko tertukar)
S = emb @ emb.T
np.fill_diagonal(S, -1.0)
pairs = []
for i in range(len(emb)):
    j = int(np.argmax(S[i]))
    pairs.append((S[i][j], person_ids[i], person_ids[j]))
pairs.sort(reverse=True)
print("\n5 PASANG CENTROID PALING MIRIP (perlu diperhatikan saat evaluasi):")
for s, a, b in pairs[:5]:
    print("  {:.4f}  {}  <->  {}".format(s, a[:38], b[:38]))

# sebaran similarity Train: genuine vs impostor (preview singkat; kalibrasi penuh di nb 04)
df_train = df[df["split"] == "train"]
idx_person = {p: i for i, p in enumerate(person_ids)}
genuine, impostor = [], []
for row in df_train.itertuples(index=False):
    e = row.emb
    if np.linalg.norm(e) < 0.5:
        continue
    i = idx_person[row.person]
    s = emb @ e
    genuine.append(s[i])
    s2 = s.copy()
    s2[i] = -1.0
    impostor.append(s2.max())
genuine = np.array(genuine)
impostor = np.array(impostor)
print("\nsimilarity Train (preview):")
print("  genuine  n={:6d} mean {:.4f}  p05 {:.4f}  p01 {:.4f}".format(
    len(genuine), genuine.mean(), np.percentile(genuine, 5), np.percentile(genuine, 1)))
print("  impostor n={:6d} mean {:.4f}  p99 {:.4f}  p95 {:.4f}  maks {:.4f}".format(
    len(impostor), impostor.mean(), np.percentile(impostor, 99), np.percentile(impostor, 95), impostor.max()))

assert emb.shape == (69, EMB_DIM), "gallery harus 69 x 512"
assert np.allclose(np.linalg.norm(emb, axis=1), 1.0, atol=1e-4), "centroid belum ternormalisasi"
assert len(set(person_ids)) == 69, "ada person id duplikat"
assert meta["jumlah_orang"] == 69, "metadata tidak sinkron dengan gallery"

# visualisasi PCA centroid
try:
    from sklearn.decomposition import PCA
    X = emb
    Y = PCA(n_components=2, random_state=SEED).fit_transform(X)
    fig, ax = plt.subplots(figsize=(9, 7))
    warna = {"SK7A": "tab:blue", "SKU7A": "tab:orange", "SKPROF": "tab:green",
             "SKALIHPROGRAM": "tab:red", "P1": "tab:purple", "-": "tab:gray"}
    kelas = [o["kelas"] for o in meta["orang"]]
    for k in sorted(set(kelas)):
        idx = [i for i, v in enumerate(kelas) if v == k]
        ax.scatter(Y[idx, 0], Y[idx, 1], s=26, alpha=0.75, label="{} (n={})".format(k, len(idx)))
    for i, p in enumerate(person_ids):
        ax.annotate(str(i), (Y[i, 0], Y[i, 1]), fontsize=6, alpha=0.6)
    ax.set_title("PCA centroid 69 orang (indeks = urutan person_ids di gallery.npz)")
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("visualisasi PCA dilewati:", e)

print("\nOK: semua pemeriksaan SEL 7 lolos.")

In [ ]:
# SEL 8 — Manajemen aktif/nonaktif per orang (FR-8: tanpa bangun ulang gallery)
# Contoh: hapus sementara 1 orang dari daftar aktif TANPA menghitung ulang embedding.
# Ubah daftar di bawah sesuai kebutuhan (kosong = semua aktif).
DAFTAR_NONAKTIF = []          # contoh: ["Pani Permatasari"]


def terapkan_aktif(gallery_npz, meta_json, nonaktif):
    g = np.load(gallery_npz, allow_pickle=False)
    emb = g["embeddings"]
    person_ids = g["person"].tolist()
    aktif = g["aktif"].copy()
    with open(meta_json, encoding="utf-8") as fh:
        meta = json.load(fh)

    for p in nonaktif:
        if p in person_ids:
            aktif[person_ids.index(p)] = False
        for o in meta["orang"]:
            if o["person"] == p:
                o["aktif"] = False

    np.savez_compressed(gallery_npz, embeddings=emb,
                        person=np.array(person_ids, dtype="U"),
                        aktif=aktif, dim=g["dim"], model=g["model"],
                        img_size=g["img_size"], dibuat=g["dibuat"])
    meta["jumlah_aktif"] = int(aktif.sum())
    with open(meta_json, "w", encoding="utf-8") as fh:
        json.dump(meta, fh, ensure_ascii=False, indent=2)
    return int(aktif.sum())


if DAFTAR_NONAKTIF:
    n_aktif = terapkan_aktif(GALLERY_NPZ, GALLERY_META, DAFTAR_NONAKTIF)
    print("dinonaktifkan        :", DAFTAR_NONAKTIF)
    print("sisa orang aktif    :", n_aktif, "/ 69")
else:
    g = np.load(GALLERY_NPZ, allow_pickle=False)
    print("tidak ada yang dinonaktifkan; aktif = {}/{}".format(int(g["aktif"].sum()), len(g["aktif"])))
print("gallery tetap sama ukurannya (69 x 512) — hanya flag aktif yang berubah (FR-8).")

In [ ]:
# SEL 9 — Ringkasan
g = np.load(GALLERY_NPZ, allow_pickle=False)
ringkas = {
    "waktu_eksekusi": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": MODEL_NAME,
    "embedding_dim": int(g["dim"][0]),
    "img_size": int(g["img_size"][0]),
    "split_dipakai": "train",
    "jumlah_centroid": int(g["embeddings"].shape[0]),
    "jumlah_aktif": int(g["aktif"].sum()),
    "file_embedding_total": int(len(df)),
    "split_file": os.path.basename(SPLIT_FILE),
}
for k, v in ringkas.items():
    print("{:<22}: {}".format(k, v))

print("\nArtefak untuk notebook 04 & 05:")
print("  1. gallery.npz         :", GALLERY_NPZ)
print("  2. gallery_meta.json   :", GALLERY_META)
print("  3. embeddings_cache.npz:", CACHE_NPZ)
print("\nLanjut -> 04_kalibrasi_threshold.ipynb")
print("(pakai split='val' untuk sweep threshold, FAR <= 1%)")

# Catatan untuk notebook 04 & 051. **Gallery dibangun dari `split='train'` saja.** Val dan Test tidak boleh ikut membentuk centroid — jika ikut, ambang threshold dan akurasi test jadi palsu.2. **`embeddings_cache.npz` berisi seluruh 17.680 embedding** (train/val/test) lengkap dengan kolom `split`. Notebook 04 & 05 memakainya agar tidak perlu menghitung ulang (~3 menit hemat).3. **Ambang batas rentang awal** (belum final — ditentukan di notebook 04): similarity Train ke centroid sendiri (genuine) p01 ≈ lihat output SEL 7, sedangkan impostor p99 juga di output SEL 7. Ambang yang masuk akal berada di antara keduanya.4. **Menambah orang baru (FR-4):** taruh foto baru di folder terpisah → jalankan notebook 01 & 02 untuk grup barunya → jalankan notebook 03. **Tidak perlu training ulang.**5. **Menonaktifkan orang (FR-8):** isi `DAFTAR_NONAKTIF` di SEL 8. Embedding tidak dihitung ulang.